# ShowPtycho: NVIDIA CUDA — advanced — metadata calibration

Load your local H5/QEM data, browse with Show4DSTEM, fit SSB, then review
and save the calibration. No aberration values are required.

Install the editable bobleesj GPU and Widget forks using the
[setup guide](setup/README.md). No data are downloaded.

In [ ]:
from pathlib import Path

from quantem.gpu import DirectPtycho, detector, io, screening
import quantem.widget as qw

qw.profile()
device = "cuda:0"
source = Path("scan_master.h5")  # Your local acquisition.
save_dir = "ssb-results-cuda-advanced"

search = "standard"
trials = None  # Standard: 200 initial trials; broad: 400 additional trials.
refinement = "newton"


## Check acquisition suitability

The default tilt/depth fit requires measured spatial frequencies in its fitting band (0.2–0.9 Å⁻¹). Coarsely sampled scans may not reach that band even when metadata are complete. The fitter stops with an explanation; Auto upsampling cannot recover missing measured frequencies.

For such an acquisition, explicitly replace the fitting call with ordinary SSB:

```python
fit = ptycho.find_aberrations(
    search="primary", tilt=False, objective="phase_variance",
)
```

This does not fit tilt/depth or broad higher-order aberrations. In the advanced notebook, skip tilt/depth landscapes and the four-case joint-tilt comparison for this alternative.

## Load data

Keep the master, detector shards and matching microscope metadata together. Recorded calibration is used without manual overrides. Missing voltage or scan sampling must be supplied from an independent calibration; use the manual notebook if necessary.

In [ ]:
data = io.load(source, device=device)
data.calibration

## Browse with Show4DSTEM

Move across the scan to inspect diffraction patterns. Drag the virtual detector
or choose BF/ADF presets to inspect specimen regions. ShowPtycho below reuses
this same loaded acquisition; browsing does not crop or bin its data.

In [ ]:
browse = qw.Show4DSTEM(data)
browse

## Use acquisition calibration

Recorded detector angular sampling is retained. If the convergence semiangle is missing, screening estimates it from the measured bright-field disk and recorded angular sampling. This depends on the detector calibration being correct. If both are missing, use the manual notebook. The following output identifies the estimate when used.

In [ ]:
products = screening.prepare(data)
products.metadata.get("semiangle_mrad_source", "Check the recorded semiangle in the calibration table")

## Inspect the mean diffraction pattern

Screening reuses the loaded acquisition and provides BF, DF, ADF and mean diffraction arrays. Inspect the disk before fitting.

In [ ]:
bf = products.bright_field
df = products.dark_field
adf = products.annular_dark_field
mean_dp = products.mean_diffraction_pattern
qw.Show2D(mean_dp, cmap="inferno")

Use `columns=2` for two images across, or `rows=2` for two down.
`size="medium"` sets each panel; `"small"`, `"large"` and `"xlarge"` are available.
The same `size`, `cmap` and `save` names apply to `detector_fit.show`,
`detector.show` and static SSB views. `rows` and `columns` apply to galleries;
the detector-fit plot has one panel. The PNG files save beside this notebook.
ShowPtycho retains its own Color control and responsive layout; `save_dir`
stores calibration, separately from these static figure exports.

For independent colormaps and histogram limits, use the retained scan images:

```python
images = qw.Show2D(
    [bf, df, adf], labels=["BF", "DF", "ADF"],
    size="medium", columns=2, link_contrast=False,
    cmap=["gray", "inferno", "inferno"],
)
images
```

Select a panel to adjust its contrast; `images.save_image("bf.png", idx=0)`
saves its displayed colors. The arrays retain their original intensities.
For a static individual image or mean diffraction pattern, use core QuantEM:

```python
from quantem.core.visualization import show_2d
show_2d(mean_dp, title="Mean diffraction", norm="log_auto", cmap="inferno")
```

For custom collection angles, use
`detector.adf(data, inner_angle_mrad=50, outer_angle_mrad=100)`.
Angles use the recorded convergence semiangle; choose a range covered by your detector.


## Fit and reconstruct automatically

The default uses **200 TPE trials for the initial optics, tilt and depth fit**,
then jointly refines all aberrations through fifth order with Newton.
`search="broad"` adds 400 fifth-order TPE trials before the same refinement.

Fitting uses native scan sampling and all valid bright-field pixels.
**Auto** sets the displayed reconstruction sampling from the calibrated bandwidth.
A supplied rotation is retained; estimated rotation is reviewed in the initial fit.
Phase polarity alone does not independently prove the rotation.

More fitted terms do not establish resolution or physical probe aberrations.
Newton can stop early; reaching its evaluation limit does not prove convergence.

Keep `trials=None` for these defaults. Set an integer to change the initial
budget in `standard`, or the extra exploration budget in `broad`.
`refinement="nelder-mead"` is available for comparison.


The initial phase appears while fifth-order refinement continues; the same view updates when finished. Use `show=False` to fit without a notebook preview.


In [ ]:
ptycho = DirectPtycho(
    data, backend="cuda",
    semiangle_mrad=products.metadata.get("semiangle_mrad"),
)
fit = ptycho.find_aberrations(search=search, trials=trials, refinement=refinement)
result = ptycho.run(fit, upsampling_factor="auto")
result.report()

## Compare rotation and defocus

This separate diagnostic uses four **primary** fits: two rotations × two
C10 signs, fitting low-order optics, tilt and depth. It preserves the active
broad or standard fifth-order fit. Phase images use
matched contrast and Auto sampling; histograms use native phase pixels.

An unresolved result needs independent focus or rotation evidence.
The comparison does not establish a global optimum or validate the broad model.

In [ ]:
comparison = ptycho.compare_rotation_defocus(show=True)

## Fit landscapes and reconstructed images

These views reuse the joint fit. Landscapes vary two model parameters while
holding the others fixed; they are not uncertainty estimates or spatial maps.
Use `ptycho.show("depth")` or `ptycho.show("tilt")` for just one landscape.


In [ ]:
ptycho.show("parameters", size="medium", columns=2, cmap="inferno", save="ssb-fit.png")


### Review focus

Compare C10 offsets −5, 0 and +5 nm while keeping the fitted depth, tilt,
rotation and all other aberrations fixed. Auto sampling and matched contrast
are retained. These are conditional reconstructions, not specimen slices.


In [ ]:
ptycho.show("focus", depth_spread_nm=fit.depth_spread_nm, size="medium", columns=3, cmap="inferno")


## Review and select regions

Inspect phase and FFT in ShowPtycho. Use Local Tilt to draw, resize or remove
regions; select a fitted region to inspect its optics and sample model. Wait
for fitting to finish before saving. The full-field overview remains available.
Region coordinates use native scan pixels; rectangles need at least 32 pixels
along each axis. Regional fits take additional searches.

Sliders are optional. Display contrast and phase negation change only the view.
SSB depth spread is a model parameter, not a direct thickness measurement.

In [ ]:
viewer = ptycho.show(local_tilt=True, show_fft=True, save_dir=save_dir)
viewer

## Save the reviewed calibration

Run this after finishing your adjustments. Reopen the same acquisition with the
same `save_dir` to restore the saved model in ShowPtycho. Use a separate output
folder for each acquisition.

In [ ]:
viewer.save_calibration()

## Optional adjustments

The automatic path above needs none of these. Copy only the recipe you need
into a code cell **after the automatic fit and before opening the viewer**.
After changing the model, reopen the viewer and save the reviewed calibration.

### Review a selected region's fit and focus

For scripted comparisons, choose bounds in native scan pixels as
`(row_start, row_stop, col_start, col_stop)`, with exclusive stops and at least
32 pixels along each axis. The example selects the upper-left quarter of a
scan with at least 64 pixels per axis; edit it to select your region.

```python
bounds = (0, data.shape[0] // 2, 0, data.shape[1] // 2)
region = ptycho.select_region(bounds)
region_fit = region.find_aberrations(
    search="primary", trials=200, refinement="auto",
    check_rotation=False, compare_standard=False,
)
region.show("focus", layout="stack", size="medium", rows=1, cmap="inferno")
```

The region fits its own optics, tilt and depth jointly, retaining the full-field
rotation and detector calibration. Its image views reconstruct the **full parent
scan**, then crop; the full-field fit is unchanged. Keep `ptycho` open while using
`region`. In following cells, use `region.show("parameters")` for landscapes,
`region.show("focus", mode="static", columns=2)` for the two-column figure, or
`region.show("focus", focus_offsets_nm=(-10, 0, 10),
depth_spread_nm=region_fit.depth_spread_nm)` to compare focus at its fitted depth.

### Apply known higher-order aberrations

Both standard and broad fit through fifth order, including tilt and depth. Apply supplied higher-order values only when you have an
independent calibration; zeros replace those terms, including broad-fitted ones.

```python
higher_order = {
    # Second order
    "C21_nm": 0.0, "phi21_deg": 0.0,
    "C23_nm": 0.0, "phi23_deg": 0.0,
    # Third order
    "C30_nm": 0.0,
    "C32_nm": 0.0, "phi32_deg": 0.0,
    "C34_nm": 0.0, "phi34_deg": 0.0,
    # Fourth order
    "C41_nm": 0.0, "phi41_deg": 0.0,
    "C43_nm": 0.0, "phi43_deg": 0.0,
    "C45_nm": 0.0, "phi45_deg": 0.0,
    # Fifth order
    "C50_nm": 0.0,
    "C52_nm": 0.0, "phi52_deg": 0.0,
    "C54_nm": 0.0, "phi54_deg": 0.0,
    "C56_nm": 0.0, "phi56_deg": 0.0,
}
result = ptycho.run(
    aberrations=higher_order, upsampling_factor="auto",
)
result.report()
```

This lists every supported higher-order term, through fifth order. Zero
values disable these added corrections; replace only independently calibrated
values. Magnitudes use nm and azimuths use degrees. C30 and C50 are rotationally
symmetric and have no angle. The fitted primary optics, tilt and depth are retained.
This applies supplied terms without another search; it is not evidence that
those terms are correct. A later automatic fit must not be assumed to retain
or optimize them. The earlier rotation diagnostic describes the earlier fit.

### Optional primary search controls

Use the current default ranges explicitly when you want to edit them:

```python
search_ranges = {"C10_nm": (-30.0, 30.0), "C12_nm": (0.0, 20.0)}
fit = ptycho.find_aberrations(
    search="primary", trials=200, refinement="auto",
    search_ranges=search_ranges,
)
```

Custom ranges require `search="primary"`. Standard and broad both support
Newton or Nelder–Mead final refinement, including for supported regions.
The regional recipe above stops after the primary fit.



## Optional independent calibration

To replace the detector calibration, use the manual notebook: supply a known convergence semiangle and fit the probe disk. A disk radius alone cannot establish an angular scale. Do not estimate a semiangle from metadata and then use that same estimate as an independent recalibration.